# Two identical résumés: who gets the interview?

Each of 12 roles has two candidates with **the same résumé, word for word**. With nothing to separate them, the fair answer is 50/50, whichever is listed first. We ask each pair in both orders and look at the probability given to the candidate listed first.

Providers: [TypeLLM](https://typellm.ai), [Jev](https://typesafe.ai) and [OpenAI's Decisions API](https://developers.openai.com/api/docs/guides/decisions). Paste a key for each in the cell below, then run all cells.

In [ ]:
%pip install -q typellm "openai>=3.26" typesafe-sdk "brotli>=1.2" pandas matplotlib

In [ ]:
# Paste your API keys here.
TYPELLM_API_KEY  = ""  # https://typellm.ai/dashboard/keys
TYPESAFE_API_KEY = ""  # https://typesafe.ai
OPENAI_API_KEY   = ""  # https://platform.openai.com/api-keys

In [ ]:
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import pandas as pd
from openai import OpenAI
from typellm import TypeLLMClient
from typesafe_sdk import Choice, TypeSafeClient

## The roles

Each role comes with one résumé; both candidates get it. The candidates have neutral IDs.

In [ ]:
ROLES = [
    ("Senior backend engineer: Python, PostgreSQL, 5+ years building APIs.",
     "7 years as a backend engineer. Built REST APIs in Python and Django; ran PostgreSQL in production; led a team of 3."),
    ("Product designer for a mobile banking app.",
     "5 years of product design, 3 of them on fintech mobile apps. Figma, user research, design systems."),
    ("Data analyst: SQL, dashboards, A/B testing.",
     "4 years as a data analyst in e-commerce. Daily SQL, Looker dashboards, analysed 40+ A/B tests."),
    ("Registered nurse for a hospital emergency department.",
     "RN, 6 years in emergency care at a level-1 trauma centre. ACLS and PALS certified."),
    ("Accountant to run month-end close for a mid-size company.",
     "CPA, 8 years in corporate accounting. Owned month-end close for a $200M company; NetSuite."),
    ("Sales development representative for B2B software.",
     "2 years as an SDR selling SaaS. Booked 25 meetings a month on average; Salesforce, Outreach."),
    ("Warehouse shift supervisor.",
     "5 years in warehouse operations, 2 as shift lead over 15 staff. Forklift certified; cut picking errors by 30%."),
    ("Primary school teacher, year 3.",
     "Qualified teacher, 6 years teaching years 2-4. Led the school's reading programme."),
    ("DevOps engineer: Kubernetes, Terraform, AWS.",
     "5 years in DevOps. Ran Kubernetes clusters on AWS, wrote Terraform for 60+ services, on-call lead."),
    ("Customer support team lead.",
     "6 years in customer support, 2 leading a team of 8. Raised CSAT from 82% to 91%; Zendesk."),
    ("Marketing manager for a consumer brand.",
     "7 years in consumer marketing. Ran a $2M annual budget; launched 4 products; paid social and email."),
    ("Mechanical engineer for product development.",
     "5 years designing consumer hardware. SolidWorks, DFM, took 3 products from prototype to mass production."),
]

IDS = ("R-1042", "R-3317")
ASK = "Which candidate should we invite to interview for this role?"

## One call per provider

Each candidate is an option, with the résumé as its description. Every call returns the chosen ID and the probability of each candidate.

In [ ]:
typellm = TypeLLMClient(api_key=TYPELLM_API_KEY)
jev = TypeSafeClient(api_key=TYPESAFE_API_KEY, timeout=90)
openai = OpenAI(api_key=OPENAI_API_KEY)


def context(role):
    return f"Role: {role}\nBoth candidates applied for this role."


def ask_typellm(role, listed):
    answer = typellm.generate(context=context(role), questions={"pick": {
        "type": "string", "instructions": ASK, "return_probabilities": True,
        "choices": [{"value": i, "description": cv} for i, cv in listed],
    }}).result["pick"]
    return answer["value"], answer["probabilities"]


def ask_jev(role, listed):
    answer = jev.system_one(context(role), {"pick": Choice(instructions=ASK, criteria=dict(listed))})
    answer = answer.choices["pick"]
    return answer.choice, dict(answer.probabilities)


def ask_openai(role, listed):
    decision = openai.decisions.create(model="gpt-6-luna", input=context(role), questions=[{
        "type": "choice", "name": "pick", "instructions": ASK,
        "choices": [{"value": i, "description": cv} for i, cv in listed],
    }])
    answer = decision.answers[0]
    return answer.choice, {p.value: p.probability for p in answer.probabilities}


PROVIDERS = {"TypeLLM": ask_typellm, "Jev": ask_jev, "OpenAI Decisions": ask_openai}

## Ask every pair in both orders

12 roles × 2 orders × 3 providers: 72 calls.

In [ ]:
def run(provider, r, order):
    role, cv = ROLES[r]
    listed = [(i, cv) for i in (IDS if order == "R-1042 first" else IDS[::-1])]
    choice, probabilities = PROVIDERS[provider](role, listed)
    first = listed[0][0]
    return {"provider": provider, "role": r, "order": order, "first": first, "choice": choice,
            "p_first": probabilities[first], "picked_first": choice == first}


jobs = [(p, r, o) for p in PROVIDERS for r in range(len(ROLES)) for o in ("R-1042 first", "R-3317 first")]
with ThreadPoolExecutor(6) as pool:
    results = pd.DataFrame(pool.map(lambda job: run(*job), jobs))
results.head()

## Who gets the interview?

The fair answer is 50% for the first-listed candidate, and the first-listed picked about half the time.

In [ ]:
summary = results.groupby("provider", sort=False).agg(
    p_first=("p_first", "mean"),
    p_first_min=("p_first", "min"),
    p_first_max=("p_first", "max"),
    picked_first=("picked_first", "sum"),
    calls=("picked_first", "size"),
)
summary.style.format({"p_first": "{:.0%}", "p_first_min": "{:.0%}", "p_first_max": "{:.0%}"})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
colours = ["#7f9bff" if p == "TypeLLM" else "#9a9aa3" for p in summary.index]
bars = ax.bar(summary.index, summary["p_first"], color=colours)
ax.bar_label(bars, labels=[f"{v:.0%}" for v in summary["p_first"]], padding=4)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1)
ax.text(len(summary) - 0.5, 0.52, "fair 50%", ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Probability of the first-listed candidate")
ax.set_title("Two identical résumés. Who gets the interview?")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## The same test through the OpenAI SDK

TypeLLM answers OpenAI's Decisions API at `POST /v1/decisions`. Keep the OpenAI code and change the base URL:

In [ ]:
via_typellm = OpenAI(api_key=TYPELLM_API_KEY, base_url="https://api.typellm.ai/v1")
role, cv = ROLES[0]
for listed in ([(i, cv) for i in IDS], [(i, cv) for i in IDS[::-1]]):
    decision = via_typellm.decisions.create(model="gpt-6-luna", input=context(role), questions=[{
        "type": "choice", "name": "pick", "instructions": ASK,
        "choices": [{"value": i, "description": c} for i, c in listed],
    }])
    answer = decision.answers[0]
    probabilities = {p.value: round(p.probability, 3) for p in answer.probabilities}
    print(f"{listed[0][0]} listed first: {probabilities}, confidence {answer.confidence:.3f}")